# Phase 1 – Data understanding
## Step 0: Load all tables

In [1]:
import pandas as pd
import os

DATA = r"C:\Users\start\OneDrive\Desktop\HVIA-TASK3\Cottonil_project\COTTONIL\COTTONIL_Data\02_TABLES"
pd.set_option("display.max_columns", 50)

In [2]:
tables = {}
for f in sorted(os.listdir(DATA)):
    if f.endswith(".csv"):
        tables[f.replace(".csv", "")] = pd.read_csv(os.path.join(DATA, f), low_memory=False)

overview = pd.DataFrame([
    {"table": n, "rows": len(df), "columns": df.shape[1]}
    for n, df in tables.items()
]).sort_values("rows", ascending=False).reset_index(drop=True)
print("Total rows:", f"{overview.rows.sum():,}")
overview

Total rows: 2,931,843


,table,rows,columns
0,sales_order_items,1138536,13
1,sales_orders,540000,13
2,sales_payments,540000,6
3,monthly_stock,142784,7
4,inventory_transfer_items,109614,4
5,purchase_order_items,103334,7
6,delivery_shipments,97660,8
7,customers,85000,6
8,operating_expenses,47000,7
9,purchase_orders,41500,9


## Step 1: `branches`

In [3]:
b = tables["branches"]
print(b.shape)
b.head()

(80, 7)


,branch_id,branch_name_ar,governorate_ar,city_ar,address_ar,retail_location_type,opening_date
0,BR001,الزمالك,القاهرة,الزمالك,شارع 26 يوليو، الزمالك,Point of sale / reseller,9/14/2019
1,BR002,السيدة زينب,القاهرة,السيدة زينب,229 شارع بورسعيد، السيدة زينب,Point of sale / reseller,2/19/2019
2,BR003,القطامية داون تاون,القاهرة,القطامية,شارع التسعين، مول داون تاون، القطامية,Retail branch,2/12/2020
3,BR004,الرحاب مول,القاهرة,الرحاب,المرحلة الثالثة، مول الرحاب,Point of sale / reseller,8/14/2019
4,BR005,المعادي جراند مول,القاهرة,المعادي,شارع 250، جراند مول، المعادي,Retail branch,11/16/2018


In [4]:
print("Empty cells:", b.isna().sum().sum())
print("Duplicate branch_id:", b.branch_id.duplicated().sum())
print("Unique governorates:", b.governorate_ar.nunique(), "| Unique cities:", b.city_ar.nunique())

Empty cells: 0
Duplicate branch_id: 0
Unique governorates: 21 | Unique cities: 65


In [5]:
print(b.retail_location_type.value_counts(), "\n")
b.governorate_ar.value_counts().to_frame("branches")

retail_location_type
Retail branch               52
Point of sale / reseller    28
Name: count, dtype: int64 



,branches
governorate_ar,
القاهرة,21
الإسكندرية,10
الجيزة,6
الدقهلية,4
البحيرة,4
الغربية,4
المنيا,4
الشرقية,3
المنوفية,3


In [6]:
b["opening_date"] = pd.to_datetime(b.opening_date, format="%m/%d/%Y")
first_sale = pd.to_datetime(tables["sales_orders"].groupby("branch_id").order_date.min())

check = b[["branch_id", "branch_name_ar", "opening_date"]].copy()
check["first_sale"] = check.branch_id.map(first_sale)
check["sold_before_opening_days"] = (check.opening_date - check.first_sale).dt.days

print("Opening dates range:", check.opening_date.min().date(), "->", check.opening_date.max().date())
print("First sale range:   ", check.first_sale.min().date(), "->", check.first_sale.max().date())
print("Branches that sold BEFORE their opening date:", (check.sold_before_opening_days > 0).sum(), "of", len(check))
check.sort_values("sold_before_opening_days", ascending=False).head()

Opening dates range: 2018-11-06 -> 2020-08-29
First sale range:    2018-10-19 -> 2018-10-22
Branches that sold BEFORE their opening date: 80 of 80


,branch_id,branch_name_ar,opening_date,first_sale,sold_before_opening_days
16,BR017,زهراء المعادي,2020-08-29,2018-10-19,680
40,BR041,طلخا,2020-08-26,2018-10-19,677
77,BR078,شرق البندر,2020-08-10,2018-10-19,661
11,BR012,البطراوي,2020-08-07,2018-10-20,657
55,BR056,سقالة,2020-07-26,2018-10-19,646


In [7]:
print(b[b.branch_name_ar.duplicated(keep=False)][["branch_id", "branch_name_ar", "governorate_ar", "address_ar"]], "\n")

for t, col in [("sales_orders", "branch_id"), ("employees", "branch_id"), ("inventory_transfers", "destination_branch_id")]:
    missing = (~tables[t][col].isin(b.branch_id)).sum()
    print(f"{t}.{col}: {tables[t][col].nunique()} branches used, {missing} codes not found in branches")

dd = tables["data_dictionary"]
print("\nColumns in dictionary but NOT in file:",
      set(dd[dd.table_name == "branches.csv"].field_name) - set(tables["branches"].columns))

   branch_id branch_name_ar governorate_ar                address_ar
41     BR042      سعد زغلول    الإسماعيلية        190 شارع سعد زغلول
56     BR057      سعد زغلول         الفيوم  ممر أبو عميرة، سعد زغلول 

sales_orders.branch_id: 80 branches used, 0 codes not found in branches
employees.branch_id: 80 branches used, 0 codes not found in branches
inventory_transfers.destination_branch_id: 80 branches used, 0 codes not found in branches

Columns in dictionary but NOT in file: {'source_url'}
